# Lab 10 — Natural Language Programming:
**ARTI 303 — Programming for AI**

**Name:** _[Noran Almohsin]_  **Student ID:** _[2250003920]_  **Section:** _[AI01]_

---

### What this lab is really about

Your team wants to flag **negative movie reviews**. Before any model sees
the text, someone has to clean it: strip the noise, split it into words,
throw away the useless ones. Every one of those steps is a **decision**.
Some help. Some quietly **change what a review means**, and nothing warns
you when they do.

So this lab doesn't just teach the tools. It gives you a small **test**,
11 reviews whose sentiment we know, and **measures** what each cleaning
step does to them. You will watch the score go up and down as you add steps,
and by the end you'll know which steps to trust.

| Part | The step | The question |
|---|---|---|
| 1 | Cleaning | how much noise is too much to remove? |
| 2 | Tokenization | what exactly is a "word" to a computer? |
| 3 | Stop words | can you drop common words without flipping meaning? |
| 4 | Lemmatization | how do you match `liked` with `like`? |
| 5 | The full pipeline | in what order, and does it work? |
| 6 | Modern tokenizers | how do today's models turn text into numbers? |

After Part 6 there is a short AI checkpoint and a list of resources.

---

## Setup

Same course folder and environment as before. (Part 6 downloads from the
internet, so it works best on a good connection. Google Colab is a fine
place to run this lab too.)

```
arti303_lab/
├── .venv/
├── requirements.txt
├── lab01.ipynb ... lab09.ipynb
├── lab10.ipynb        <- this lab
├── data/
└── ...
```

**1.** Open `arti303_lab` in VS Code (`File → Open Folder`) — the folder
itself, not a lab file inside it.

**2.** Get the environment ready, in this order:

- Look for a `.venv` folder in the file list. If it isn't there, create
  it once: `python -m venv .venv`
- Activate it:

```bash
source .venv/bin/activate# macOS / Linux / Git Bash
.venv\Scripts\Activate.ps1         # Windows PowerShell
          ```

- Check your prompt starts with `(.venv)`. If it shows `(base)`, you're
  in Anaconda's environment and packages will install to the wrong place.

**3.** Install today's libraries and record them:

```bash
python -m pip install nltk transformers
python -m pip freeze > requirements.txt
```

**4.** Point this notebook at the same environment:
`Cmd+Shift+P` (macOS) / `Ctrl+Shift+P` (Windows) → **Python: Select
Interpreter** → the entry showing `.venv`. Then restart the kernel.

### Check your setup

In [1]:
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")    # a harmless notebook message

problems = []

if "google.colab" in sys.modules or os.path.exists("/kaggle"):
    print("[ok]   Running on Colab or Kaggle (no virtual environment needed)")
elif sys.prefix != sys.base_prefix:
    print("[ok]   Running inside a virtual environment")
else:
    print("[!]    NOT in a virtual environment")
    problems.append("Select the .venv interpreter, then restart the kernel")
print("       using:", sys.executable)
print()

for import_name, pip_name in [("nltk", "nltk"), ("transformers", "transformers"),
                              ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        module = __import__(import_name)
        print(f"[ok]   {pip_name} {module.__version__}")
    except ImportError:
        print(f"[!]    {pip_name} is not installed in this environment")
        problems.append(f"Run: python -m pip install {pip_name}")

if os.path.exists("requirements.txt"):
    names = [l.split("==")[0].strip().lower() for l in open("requirements.txt").read().splitlines() if l.strip()]
    for name in ["nltk", "transformers"]:
        if name in names:
            print(f"[ok]   {name} is recorded in requirements.txt")
        else:
            print(f"[!]    {name} is missing from requirements.txt")
            problems.append("Run: python -m pip freeze > requirements.txt")
    if len(names) > 100:
        print("[!]    requirements.txt has", len(names), "packages - that looks like Anaconda's base")
        problems.append("Delete it, activate .venv, freeze again")

print()
print("Fix these first:" if problems else "Setup looks good.")
for item in problems:
    print("  -", item)

[ok]   Running inside a virtual environment
       using: /Users/noran/Documents/GitHub/arti303/.venv/bin/python

[ok]   nltk 3.10.4
[ok]   transformers 5.22.2
[ok]   scikit-learn 1.9.1
[ok]   torch 2.15.0
[ok]   nltk is recorded in requirements.txt
[ok]   transformers is recorded in requirements.txt

Setup looks good.


NLTK keeps its dictionaries and models in separate downloads. Fetch the
ones this lab needs (about a minute, once):

In [2]:
import nltk

for resource in ["punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    ok = nltk.download(resource, quiet=True)
    print(f"[{'ok' if ok else '!'}] {resource}")

[nltk_data] Error loading punkt_tab: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


[!] punkt_tab


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


[!] stopwords


[nltk_data] Error loading wordnet: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


[!] wordnet


[nltk_data] Error loading omw-1.4: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


[!] omw-1.4
[!] averaged_perceptron_tagger_eng


[nltk_data] Error loading averaged_perceptron_tagger_eng: <urlopen
[nltk_data]     error [SSL: CERTIFICATE_VERIFY_FAILED] certificate
[nltk_data]     verify failed: unable to get local issuer certificate
[nltk_data]     (_ssl.c:1082)>


---
### The test: 11 reviews we can grade

Here are 11 short reviews, each labelled **positive (+1)** or **negative
(−1)**. They are hand-written, and they include the awkward cases real
reviews have: HTML left over from a web page, contractions like `wasn't`,
and negations like `not good`.

To grade a cleaning pipeline we need *something* that reads the cleaned
words and decides if a review is positive or negative. Here is a
deliberately **simple** scorer: it counts positive words minus negative
words, and a **negation** (`not`, `no`, `never`, `n't`) right before a word
**flips** it.

We are not testing the scorer. We are using it as a **fair judge** to
measure what each cleaning step does to the reviews' meaning.

In [3]:
from nltk.tokenize import word_tokenize

REVIEWS = [
    ("I loved it and enjoyed every minute.", 1),
    ("I disliked it and regretted watching.", -1),
    ("Not good at all.", -1),
    ("Not bad, I would watch it again.", 1),
    ("No good scenes, no good acting.", -1),
    ("It wasn't bad, actually quite good.", 1),
    ("I didn't like it, it wasn't good.", -1),
    ("Excellent story and wonderful music.", 1),
    ("Boring and terrible. A waste of time.", -1),
    ("<br />Great fun, I loved the ending!", 1),
    ("Awful. Poor acting and a bad script.", -1),
]

POSITIVE = {"good", "great", "love", "enjoy", "excellent", "wonderful", "like"}
NEGATIVE = {"bad", "boring", "terrible", "dislike", "awful", "poor", "waste", "regret"}
NEGATIONS = {"not", "no", "never", "n't"}


def sentiment_score(tokens):
    """Positive words minus negative words. A negation right before a word flips it."""
    score = 0
    for i, token in enumerate(tokens):
        polarity = 1 if token in POSITIVE else -1 if token in NEGATIVE else 0
        if polarity and i > 0 and tokens[i - 1] in NEGATIONS:
            polarity = -polarity
        score += polarity
    return score


def accuracy_of(to_tokens):
    """The share of reviews whose sign (positive/negative) the pipeline gets right.
    to_tokens is a function: review text in, list of tokens out."""
    right = 0
    for text, label in REVIEWS:
        score = sentiment_score(to_tokens(text))
        right += (score > 0) - (score < 0) == label
    return right / len(REVIEWS)


def show_mistakes(to_tokens):
    """Print the reviews the pipeline gets wrong, with what it saw."""
    for text, label in REVIEWS:
        tokens = to_tokens(text)
        score = sentiment_score(tokens)
        if (score > 0) - (score < 0) != label:
            print(f"wrong (truth {label:+d}, score {score:+d}): {text}")
            print(f"      the scorer saw: {tokens}")


def raw_tokens(text):
    return word_tokenize(text.lower())


print("with NO cleaning at all, only lowercase + splitting into words:")
print(f"accuracy = {accuracy_of(raw_tokens):.2f}")
print()
show_mistakes(raw_tokens)

with NO cleaning at all, only lowercase + splitting into words:
accuracy = 0.82

wrong (truth +1, score +0): I loved it and enjoyed every minute.
      the scorer saw: ['i', 'loved', 'it', 'and', 'enjoyed', 'every', 'minute', '.']
wrong (truth -1, score +0): I disliked it and regretted watching.
      the scorer saw: ['i', 'disliked', 'it', 'and', 'regretted', 'watching', '.']


That is the **baseline**: 0.82. Two reviews are missed, because their
sentiment sits in words like `loved` and `disliked`, which the scorer
doesn't recognise (Part 4 will fix that).

From here, every step we add can be judged against that number.

---
# Part 1 · Cleaning: how much noise is too much?

Raw text is noisy: HTML tags, odd symbols, inconsistent capitals. Cleaning
removes what isn't relevant, and matters most for web-scraped text and
social media. **Regular expressions** (`re`) do the work. A first attempt:

In [4]:
import re


def clean_html(text):
    text = re.sub("<.*?>", "", text)                  # remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)        # remove special characters
    return text


print(clean_html("<p>This is a <b>sample</b> text!</p>"))
print("Lowercase:", "The Quick Brown Fox".lower())

This is a sample text
Lowercase: the quick brown fox


Looks fine. Now see what that same cleaner does to a review with a
contraction:

In [5]:
sample = "It wasn't bad, actually quite good."

print("no cleaning:    ", word_tokenize(sample.lower()))
print("clean_html:     ", word_tokenize(clean_html(sample).lower()))
print()
print("score without cleaning:", sentiment_score(word_tokenize(sample.lower())), "  (the review is positive)")
print("score after clean_html:", sentiment_score(word_tokenize(clean_html(sample).lower())))

no cleaning:     ['it', 'was', "n't", 'bad', ',', 'actually', 'quite', 'good', '.']
clean_html:      ['it', 'wasnt', 'bad', 'actually', 'quite', 'good']

score without cleaning: 2   (the review is positive)
score after clean_html: 0


The regular expression removed the **apostrophe** from `wasn't`. The
tokenizer used to split it into `was` + `n't`, and `n't` is the negation
that flips `bad` into something good. Now it sees `wasnt` (a word that
doesn't exist) and the negation is gone, so the positive review scores
**0**.

> **Every cleaning rule has a cost.** "Remove special characters" sounds
> harmless until it removes the characters that carry meaning.

A better rule keeps the apostrophe. In a regular expression, `[^a-z]` means
"anything that is **not** a lowercase letter", so
`[^a-zA-Z0-9\s']` means "anything that is not a letter, digit, space or
apostrophe".

### Exercise 1.1
Write `clean_text(text)`. It should:

1. remove HTML tags
2. remove special characters, **keeping letters, digits, spaces and
   apostrophes**
3. convert to lowercase
4. squeeze runs of spaces into one, and trim the ends

In [6]:
def clean_text(text):
    """Remove HTML and special characters (keeping apostrophes), lowercase, and tidy the spaces."""
    # TODO
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", "", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text



In [7]:
tests = {
    "<p>This is a <b>sample</b> text!</p>": "this is a sample text",
    "I can't wait!!": "i can't wait",
    "<div>Open 24/7: call now!</div>": "open 247 call now",
}
for raw, expected in tests.items():
    got = clean_text(raw)
    assert got is not None, "clean_text returned None"
    assert got == expected, f"clean_text({raw!r}) gave {got!r}, expected {expected!r}"
print("Exercise 1.1 correct.")
print("wasn't survives:", word_tokenize(clean_text("It wasn't bad")))

Exercise 1.1 correct.
wasn't survives: ['it', 'was', "n't", 'bad']


---
# Part 2 · Tokenization: what is a "word"?

A computer sees a string of characters, not words. **Tokenization** splits
text into units called **tokens**. It is the first step of nearly every
NLP pipeline. NLTK provides two kinds:

- `sent_tokenize`: text into **sentences**
- `word_tokenize`: text into **words** (and punctuation marks)

In [8]:
from nltk.tokenize import sent_tokenize

text = "I study Programming for AI. Currently, I'm studying NLP."

print("sentences:")
for sentence in sent_tokenize(text):
    print("  ", sentence)

print()
print("words:", word_tokenize(text))

sentences:
   I study Programming for AI.
   Currently, I'm studying NLP.

words: ['I', 'study', 'Programming', 'for', 'AI', '.', 'Currently', ',', 'I', "'m", 'studying', 'NLP', '.']


Notice three things:

- punctuation marks (`.` and `,`) become tokens of their own
- `I'm` is split into **two** tokens, `I` and `'m`. That is exactly why
  `wasn't` became `was` + `n't` in Part 1, and why our scorer can find the
  negation
- each full stop stays with the sentence it ends, so `sent_tokenize`
  returns whole sentences

**What counts as a word is a choice**, and later steps depend on it.

### Exercise 2.1
For the `paragraph` below, compute:

- `n_sentences`: the number of sentences
- `words`: a list of the **words only** (lowercase them first, then drop
  the tokens that aren't alphabetic, such as punctuation; `str.isalpha()`
  helps)
- `n_words`: how many words there are

In [9]:
paragraph = "Natural language processing is fun. Computers read numbers, not words! Do you study NLP?"

n_sentences = len(sent_tokenize(paragraph))
words = [w for w in word_tokenize(paragraph.lower()) if w.isalpha()]
n_words = len(words)
# TODO: fill in all three


In [10]:
assert n_sentences is not None and words is not None and n_words is not None, "fill in all three"
assert n_sentences == 3, f"there are 3 sentences, you have {n_sentences}"
assert all(w.isalpha() and w == w.lower() for w in words), "words should be lowercase, alphabetic tokens only"
assert n_words == len(words) == 14, f"there are 14 words (punctuation does not count), you have {len(words)}"
assert words[:3] == ["natural", "language", "processing"], "keep the words in their original order"
print("Exercise 2.1 correct.")

Exercise 2.1 correct.


---
# Part 3 · Stop words: can you drop common words safely?

**Stop words** are very common words (`the`, `is`, `in`, `and`, `a`) that
carry little meaning on their own. Removing them shrinks your data and
focuses on the words that matter. NLTK ships an English list:

In [11]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
print("words in NLTK's English list:", len(stop_words))

sentence = "The cat is sitting on the mat."
words_in_sentence = word_tokenize(sentence.lower())
filtered_words = [word for word in words_in_sentence if word not in stop_words]

print("original:", words_in_sentence)
print("filtered:", filtered_words)

words in NLTK's English list: 198
original: ['the', 'cat', 'is', 'sitting', 'on', 'the', 'mat', '.']
filtered: ['cat', 'sitting', 'mat', '.']


The full stop **survived**: stop-word lists contain words, not
punctuation, so punctuation needs separate handling.

That worked nicely. Now the same technique on our reviews, and the same
judge as before:

In [12]:
def tokens_without_stops(text):
    return [t for t in word_tokenize(clean_html_keep(text)) if t not in stop_words]


def clean_html_keep(text):
    text = re.sub("<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", "", text)
    return re.sub(r"\s+", " ", text.lower()).strip()


print(f"accuracy with stop words removed: {accuracy_of(tokens_without_stops):.2f}   (it was 0.82)")
print()
show_mistakes(tokens_without_stops)

accuracy with stop words removed: 0.55   (it was 0.82)

wrong (truth +1, score +0): I loved it and enjoyed every minute.
      the scorer saw: ['loved', 'enjoyed', 'every', 'minute']
wrong (truth -1, score +0): I disliked it and regretted watching.
      the scorer saw: ['disliked', 'regretted', 'watching']
wrong (truth -1, score +1): Not good at all.
      the scorer saw: ['good']
wrong (truth +1, score -1): Not bad, I would watch it again.
      the scorer saw: ['bad', 'would', 'watch']
wrong (truth -1, score +2): No good scenes, no good acting.
      the scorer saw: ['good', 'scenes', 'good', 'acting']


Accuracy **dropped**, and the reviews it now gets wrong are the ones with
**negation**. Look at `Not good at all.`: the scorer now sees only
`['good', '.']` and calls it positive.

Why? `not`, `no` and `nor` are **on the stop-word list**:

In [13]:
print("is 'not' a stop word? ", "not" in stop_words)
print("is 'no' a stop word?  ", "no" in stop_words)
print("is \"n't\" a stop word? ", "n't" in stop_words, "  (so `wasn't` kept its negation)")

is 'not' a stop word?  True
is 'no' a stop word?   True
is "n't" a stop word?  False   (so `wasn't` kept its negation)


> **Stop-word removal can delete negation, and negation changes meaning.**
> Removal is **optional**, and depends on the task. For sentiment, it's
> dangerous.

You can fix it by **keeping** certain words, and you can also add
**custom stop words** for your domain (for example `patient` in medical
text, where it appears in every document).

### Exercise 3.1
Write `remove_stopwords(tokens, keep=(), extra=())`:

- remove every token that is an English stop word,
- **except** the words listed in `keep` (those stay),
- and also remove the words listed in `extra` (your custom stop words).

Return a **new** list.

In [14]:
def remove_stopwords(tokens, keep=(), extra=()):
    """Remove stop words, except those in keep, and also remove those in extra."""
    # TODO
    return [t for t in tokens if (t not in stop_words or t in keep) and t not in extra]


In [15]:
original = ["the", "cat", "is", "sitting", "on", "the", "mat"]
assert remove_stopwords(original) == ["cat", "sitting", "mat"], "plain stop-word removal is wrong"
assert original == ["the", "cat", "is", "sitting", "on", "the", "mat"], "do not change the list you were given: return a new one"

review_tokens = ["this", "movie", "is", "not", "good"]
assert remove_stopwords(review_tokens) == ["movie", "good"], "without keep, 'not' is removed like any stop word"
assert remove_stopwords(review_tokens, keep=("not",)) == ["movie", "not", "good"], "words in keep must stay"

assert remove_stopwords(["this", "movie", "is", "good"], extra=("movie",)) == ["good"], "words in extra must also be removed"
print("Exercise 3.1 correct.")

Exercise 3.1 correct.


---
# Part 4 · Lemmatization: matching `liked` with `like`

Our baseline missed two reviews because their sentiment words are
**inflected**: `loved`, `enjoyed`, `disliked`, `regretted`. The scorer only
knows `love`, `enjoy`, `dislike`, `regret`. To match them you need each
word's **base form**.

**Lemmatization** converts a word to its base or dictionary form, its
**lemma**: `better` → `good`, `went` → `go`, `mice` → `mouse`.

It is different from **stemming**, which chops off word endings by fixed
rules and doesn't always produce a real word. Stemmers are faster;
lemmatizers are slower but more meaningful. NLTK's `WordNetLemmatizer`
looks words up in a dictionary (WordNet):

In [16]:
from nltk.stem import WordNetLemmatizer, PorterStemmer

lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

sentence = "The cats were running faster than the dogs."
tokens = word_tokenize(sentence)

print("lemmas, no POS given:", [lemmatizer.lemmatize(word) for word in tokens])

lemmas, no POS given: ['The', 'cat', 'were', 'running', 'faster', 'than', 'the', 'dog', '.']


`cats` became `cat`, but `running` stayed `running` and `were` stayed
`were`. Without a **part of speech (POS)** the lemmatizer assumes every
word is a **noun**. Tell it the POS (`"v"` verb, `"n"` noun, `"a"`
adjective):

In [17]:
print("running as a noun:", lemmatizer.lemmatize("running"))
print("running as a verb:", lemmatizer.lemmatize("running", pos="v"))
print()

words = ["studies", "was", "better", "mice"]
print("stemmer:   ", [stemmer.stem(w) for w in words])
print("lemmatizer:", [lemmatizer.lemmatize(w, pos=p) for w, p in [("studies", "n"), ("was", "v"), ("better", "a"), ("mice", "n")]])

running as a noun: running
running as a verb: run

stemmer:    ['studi', 'wa', 'better', 'mice']
lemmatizer: ['study', 'be', 'good', 'mouse']


The stemmer produced `studi` and `wa`, which aren't words. That is the
weakness of stemmers: they may not give semantically meaningful base words.

### Exercise 4.1
Lemmatize each word **with its part of speech** and store the lemmas, in
order, in `lemmas`.

In [18]:
word_pos = [("running", "v"), ("better", "a"), ("studies", "n"), ("was", "v"), ("went", "v"), ("mice", "n")]

# TODO: a list with the lemma of every word, using its part of speech
lemmas = [lemmatizer.lemmatize(word, pos=pos) for word, pos in word_pos]


In [19]:
assert lemmas is not None, "fill in lemmas"
assert lemmas == ["run", "good", "study", "be", "go", "mouse"], \
    f"expected ['run', 'good', 'study', 'be', 'go', 'mouse'], got {lemmas}: did you pass pos= for every word?"
print("Exercise 4.1 correct.")

Exercise 4.1 correct.


---
# Part 5 · The full pipeline: in what order, and does it work?

You have every step. The remaining question is **order**, and it matters
more than you'd think.

**Who supplies the part of speech?** In Exercise 4.1 you gave it by hand. In a
real pipeline NLTK's `pos_tag` works it out. But a tagger needs the **whole
sentence** for context. Watch what happens to one review if the stop words
are removed **before** tagging:

In [20]:
from nltk import pos_tag

sentence = "I loved it and enjoyed every minute."
tokens = word_tokenize(sentence.lower())
no_stops = [t for t in tokens if t not in stop_words]

print("tags, whole sentence:   ", pos_tag(tokens))
print("tags, stop words gone:  ", pos_tag(no_stops))

tags, whole sentence:    [('i', 'NN'), ('loved', 'VBD'), ('it', 'PRP'), ('and', 'CC'), ('enjoyed', 'VBN'), ('every', 'DT'), ('minute', 'NN'), ('.', '.')]
tags, stop words gone:   [('loved', 'VBN'), ('enjoyed', 'JJ'), ('every', 'DT'), ('minute', 'NN'), ('.', '.')]


With the stop words gone, `enjoyed` is tagged as an **adjective**
instead of a verb, so it would **not** be lemmatized to `enjoy`. The
tagger lost its context.

> **Rule: tag (and lemmatize) first, then remove stop words.**

Here are Parts 1 to 4 as tidy, reusable helpers. `lemmatize_tokens` uses
`pos_tag` to find each word's part of speech automatically:

In [21]:
def clean_ref(text):
    text = re.sub("<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s']", "", text)
    return re.sub(r"\s+", " ", text.lower()).strip()


def stopwords_ref(tokens, keep=()):
    stops = set(stopwords.words("english")) - set(keep)
    return [token for token in tokens if token not in stops]


def lemmatize_tokens(tokens):
    """Lemmatize, using each word's automatically detected part of speech."""
    wordnet_pos = {"J": "a", "V": "v", "N": "n", "R": "r"}
    return [lemmatizer.lemmatize(word, pos=wordnet_pos.get(tag[0], "n")) for word, tag in pos_tag(tokens)]


print(lemmatize_tokens(word_tokenize("i loved it and enjoyed every minute")))

['i', 'love', 'it', 'and', 'enjoy', 'every', 'minute']


### Exercise 5.1
Write `preprocess(text, keep=())`: raw text in, a **list of cleaned
lemmas** out. Use the helpers, in this order:

1. `clean_ref`
2. `word_tokenize`
3. `lemmatize_tokens`
4. `stopwords_ref` (passing it `keep`)

Then **measure it** with `accuracy_of`:

- `acc_naive`: your pipeline, keeping nothing
- `acc_careful`: your pipeline, keeping the negations `("not", "no", "nor")`

(`accuracy_of` wants a function from text to tokens, so for example
`accuracy_of(lambda text: preprocess(text))`.)

In [22]:
def preprocess(text, keep=()):
    """Raw text -> cleaned, lemmatized tokens, stop words removed (except those in keep)."""
    # TODO: clean -> tokenize -> lemmatize -> remove stop words
    text = clean_ref(text)
    tokens = word_tokenize(text)
    lemmas = lemmatize_tokens(tokens)
    return stopwords_ref(lemmas, keep=keep)

acc_naive = accuracy_of(lambda text: preprocess(text))  # TODO: accuracy_of(...) with preprocess, keeping nothing
acc_careful = accuracy_of(lambda text: preprocess(text, keep=("not", "no", "nor")))  # TODO: accuracy_of(...) with preprocess, keeping ("not", "no", "nor")


In [23]:
KEEP_NEGATIONS = ("not", "no", "nor")

for raw in ["<p>The cats were running faster than the dogs!</p>", "<b>Students</b> were studying machine learning.",
            "The mice went home.", "I loved it and enjoyed every minute."]:
    got = preprocess(raw)
    assert got is not None, "preprocess returned None"
    expected = stopwords_ref(lemmatize_tokens(word_tokenize(clean_ref(raw))))
    assert got == expected, (
        f"preprocess({raw!r}) gave {got}, expected {expected}. "
        "Check the order: the tagger needs the whole sentence, so lemmatize BEFORE removing stop words."
    )

kept = preprocess("This movie is not good", keep=KEEP_NEGATIONS)
assert "not" in kept, f"with keep=('not',...) the word 'not' must survive, but you got {kept}"
assert "not" not in preprocess("This movie is not good"), "without keep, 'not' is a normal stop word and should go"

assert acc_naive is not None and acc_careful is not None, "fill in acc_naive and acc_careful"
assert acc_careful > acc_naive, f"keeping the negations should help: naive {acc_naive:.2f}, careful {acc_careful:.2f}"
assert abs(acc_careful - 1.0) < 1e-9, f"the careful pipeline should get every review right, yours scored {acc_careful:.2f}"
print("Exercise 5.1 correct.")

Exercise 5.1 correct.


### Every step, measured

Now you can see what each step did to the same 11 reviews:

In [24]:
def strip_apostrophes_tokens(text):
    return word_tokenize(clean_html(text).lower())


stages = [
    ("0  baseline: lowercase + split",              raw_tokens),
    ("1  + clean that strips apostrophes",          strip_apostrophes_tokens),
    ("2  + clean that keeps apostrophes",           lambda t: word_tokenize(clean_ref(t))),
    ("3  + stop words removed (naive)",             lambda t: stopwords_ref(word_tokenize(clean_ref(t)))),
    ("4  + lemmatized (naive pipeline)",            lambda t: stopwords_ref(lemmatize_tokens(word_tokenize(clean_ref(t))))),
    ("5  careful: keep not / no / nor",             lambda t: stopwords_ref(lemmatize_tokens(word_tokenize(clean_ref(t))), keep=KEEP_NEGATIONS)),
]

for name, pipeline in stages:
    accuracy = accuracy_of(pipeline)
    print(f"{name:45} {accuracy:.2f}  {'#' * round(accuracy * 30)}")

0  baseline: lowercase + split                0.82  #########################
1  + clean that strips apostrophes            0.64  ###################
2  + clean that keeps apostrophes             0.82  #########################
3  + stop words removed (naive)               0.55  ################
4  + lemmatized (naive pipeline)              0.73  ######################
5  careful: keep not / no / nor               1.00  ##############################


Read the table from top to bottom:

- **Stripping apostrophes** hurt (step 1) and keeping them fixed it (step 2)
- **Removing all stop words** hurt the most (step 3), because it deleted the negations
- **Lemmatizing** recovered some of it (step 4)
- **Keeping the negations** got every review right (step 5)

The "standard" pipeline (steps 1 to 4) scored **worse than doing nothing**.
Without a test like this you would never have known.

### From words to numbers

A model needs **numbers**. The simplest way is **Bag-of-Words**: build a
vocabulary, then count how often each word appears in each document.
scikit-learn's `CountVectorizer` does it. (Each row is a document, and each
column is a word.)

In [25]:
from sklearn.feature_extraction.text import CountVectorizer

documents = [stopwords_ref(lemmatize_tokens(word_tokenize(clean_ref(text))), keep=KEEP_NEGATIONS) for text, _ in REVIEWS[:4]]
for text, tokens in zip([t for t, _ in REVIEWS[:4]], documents):
    print(f"{text:42} -> {tokens}")

vectorizer = CountVectorizer(analyzer=lambda tokens: tokens)
counts = vectorizer.fit_transform(documents)

print()
print("vocabulary:", sorted(vectorizer.vocabulary_))
print(counts.toarray())

I loved it and enjoyed every minute.       -> ['love', 'enjoy', 'every', 'minute']
I disliked it and regretted watching.      -> ['dislike', 'regret', 'watching']
Not good at all.                           -> ['not', 'good']
Not bad, I would watch it again.           -> ['not', 'bad', 'would', 'watch']

vocabulary: ['bad', 'dislike', 'enjoy', 'every', 'good', 'love', 'minute', 'not', 'regret', 'watch', 'watching', 'would']
[[0 0 1 1 0 1 1 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 1 0 1 0]
 [0 0 0 0 1 0 0 1 0 0 0 0]
 [1 0 0 0 0 0 0 1 0 1 0 1]]


Four reviews became a table of counts. Turn it into a **tensor** (as in
Labs 8 and 9) and it is ready for a model:

In [26]:
import torch

counts_tensor = torch.tensor(counts.toarray(), dtype=torch.float32)
print(tuple(counts_tensor.shape), counts_tensor.dtype)

(4, 12) torch.float32


Bag-of-Words ignores word **order** and treats every word as equally
important. TF-IDF weights rare words higher, and word embeddings capture
**meaning**. They are the next steps up from counts.

---
# Part 6 · How modern models tokenize

Everything so far was **classical** preprocessing, where *you* decide what
to keep. Modern language models do it differently. They ship with their own
**tokenizer**, and you hand them close to the **raw text**.

**Hugging Face** is an open-source platform for sharing AI models. It hosts
thousands of pre-trained models (the **Model Hub**) and the libraries to use
them. Its tokenizers do more than NLTK:

- they split words into **subword pieces**, so an unknown word is built from
  known parts instead of being lost
- they turn tokens into **IDs** (the numbers a model reads)
- they add **special tokens** (such as a start and an end marker)
- they **pad** short sentences so a batch has one shape, and give an
  **attention mask** saying which positions are real

The cell below loads `bert-base-uncased` from the Hugging Face Hub. It
first checks (in a few seconds) whether the Hub can be reached. If it
**can't** (no internet, or a blocked network), it switches to a small
tokenizer built on the spot, so you can keep going. The exercise works
with either.

In [27]:
import os
import urllib.request

os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "10")

from transformers import AutoTokenizer, PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers, processors, decoders


def build_local_tokenizer():
    """A small backup tokenizer (you don't need to read this)."""
    corpus = [
        "the cat sat on the mat", "the quick brown fox jumps over the lazy dog", "tokenization is fun",
        "i study programming for ai", "natural language processing helps computers understand human language",
        "ravi is happy with the new phone", "this movie is not good", "i loved it and enjoyed every minute",
        "not good at all", "this lab is about tokenizers and models", "students are learning to build models",
        "hugging face provides pretrained models", "machines process numbers not words",
        "i study programming for ai and natural language processing",
    ]
    tok = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tok.normalizer = normalizers.BertNormalizer(lowercase=True)
    tok.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    trainer = trainers.WordPieceTrainer(
        vocab_size=400, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
        initial_alphabet=list("abcdefghijklmnopqrstuvwxyz"), limit_alphabet=100,
    )
    tok.train_from_iterator(corpus, trainer)
    tok.decoder = decoders.WordPiece(prefix="##")
    tok.post_processor = processors.TemplateProcessing(
        single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
        special_tokens=[("[CLS]", tok.token_to_id("[CLS]")), ("[SEP]", tok.token_to_id("[SEP]"))],
    )
    return PreTrainedTokenizerFast(
        tokenizer_object=tok, unk_token="[UNK]", pad_token="[PAD]",
        cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]",
    )


def hub_reachable(timeout=4):
    """A quick test, so we don't wait through long retries when there is no connection."""
    try:
        urllib.request.urlopen("https://huggingface.co", timeout=timeout)
        return True
    except Exception:
        return False


def load_tokenizer():
    if hub_reachable():
        try:
            tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
            print("Loaded bert-base-uncased from the Hugging Face Hub.")
            return tokenizer
        except Exception as error:
            print(f"The Hub answered, but the download failed ({type(error).__name__}).")
    else:
        print("The Hugging Face Hub can't be reached from here.")
    print("Using a small local tokenizer instead. Everything below still works.")
    return build_local_tokenizer()


tokenizer = load_tokenizer()

The Hugging Face Hub can't be reached from here.
Using a small local tokenizer instead. Everything below still works.





### Tokens, IDs and special tokens

Notice what a modern tokenizer does with the sentence that broke our
stop-word step:

In [28]:
sentence = "Not good at all"

tokens = tokenizer.tokenize(sentence)
encoded = tokenizer(sentence)

print("tokens:  ", tokens)
print("ids:     ", encoded["input_ids"])
print("with specials:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print()
print("'not' is still there:", "not" in tokens)
print()
print("a made-up word is split into pieces:", tokenizer.tokenize("zorptastic"))

tokens:   ['not', 'good', 'at', 'all']
ids:      [2, 91, 117, 125, 202, 3]
with specials: ['[CLS]', 'not', 'good', 'at', 'all', '[SEP]']

'not' is still there: True

a made-up word is split into pieces: ['z', '##o', '##r', '##p', '##t', '##a', '##st', '##i', '##c']


`not` is **still there**. A modern tokenizer doesn't throw words away, so
**stop-word removal and lemmatization are generally unnecessary** before a
pretrained model. They are classical techniques for classical methods.

Look at the tokens that start with `##`: they **continue the previous
piece** of the same word. The made-up word isn't in any vocabulary, yet the
tokenizer can still represent it. The `with specials` line shows the extra
start and end markers added around the sentence.

### Batches, padding and the attention mask

Models take **batches**, so every sentence needs the same length. The
tokenizer pads the short ones, and the **attention mask** marks real
tokens (`1`) versus padding (`0`).

### Exercise 6.1
Encode the three `sentences` as **one padded batch of PyTorch tensors**,
in `batch` (one call to `tokenizer(...)` with `padding=True` and
`return_tensors="pt"`). Then work out `real_lengths`: for each sentence,
the number of **real** (non-padding) tokens, as a list. Hint: sum the
attention mask along each row.

In [29]:
sentences = [
    "The cat sat on the mat",
    "Tokenization is fun",
    "I study programming for AI and natural language processing",
]

# TODO: batch = all three sentences, padded, as PyTorch tensors
batch = tokenizer(sentences, padding=True, return_tensors="pt")

# TODO: real_lengths = the number of real tokens in each sentence (a list)
real_lengths = batch["attention_mask"].sum(dim=1).tolist()



In [30]:
assert batch is not None and real_lengths is not None, "fill in batch and real_lengths"
ids, mask = batch["input_ids"], batch["attention_mask"]
assert isinstance(ids, torch.Tensor), 'ask for PyTorch tensors with return_tensors="pt"'
assert isinstance(real_lengths, (list, tuple)) or (isinstance(real_lengths, torch.Tensor) and real_lengths.ndim == 1), \
    "real_lengths should be a list with one number per sentence: sum the attention mask along dim=1 (and use .tolist())"
assert ids.shape[0] == 3 and ids.shape == mask.shape, "expected one row per sentence, with input_ids and attention_mask the same shape"
assert ids.dtype == torch.int64, f"token IDs should be int64, they are {ids.dtype}"

individual = [len(tokenizer(s)["input_ids"]) for s in sentences]
assert ids.shape[1] == max(individual), 'padding=True should pad every sentence up to the longest one'
assert list(real_lengths) == individual, f"real_lengths should be {individual}: sum the attention mask along each row (dim=1)"
print("Exercise 6.1 correct.")

Exercise 6.1 correct.


The batch is a normal PyTorch tensor, so everything from Lab 9 applies.
Move it to the same device as your model with `.to(device)`:

In [31]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if batch is None:
    print("Finish Exercise 6.1 to see this.")
else:
    batch_on_device = batch.to(device)
    print("input_ids now on:", batch_on_device["input_ids"].device)
    print("decoded back:    ", tokenizer.decode(batch["input_ids"][0], skip_special_tokens=True))

input_ids now on: cpu
decoded back:     the cat sat on the mat


---
# AI checkpoint

**Pattern: measure it, don't read it.**

You have a test that grades any preprocessing. Use it on an AI's.

### Step 1
Give an AI assistant this exact prompt, unmodified:

> "Write a Python function `ai_preprocess(text)` using NLTK that
> preprocesses a movie review for sentiment analysis: lowercase it,
> tokenize it, remove stop words and punctuation, and lemmatize it. It
> should return a list of tokens."

### Step 2
Paste its function below and run it. Then **grade** it with
`accuracy_of(ai_preprocess)`, and look at what it got wrong with
`show_mistakes(ai_preprocess)`.

In [32]:

# Paste the AI assistant's function here, then run:
#
# print(accuracy_of(ai_preprocess))
# show_mistakes(ai_preprocess)

def ai_preprocess(text):
    text = text.lower()
    tokens = word_tokenize(text)
    stop_words_set = set(stopwords.words("english"))

    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words_set
    ]

    return [lemmatizer.lemmatize(word) for word in tokens]

print(accuracy_of(ai_preprocess))
show_mistakes(ai_preprocess)


0.36363636363636365
wrong (truth +1, score +0): I loved it and enjoyed every minute.
      the scorer saw: ['loved', 'enjoyed', 'every', 'minute']
wrong (truth -1, score +0): I disliked it and regretted watching.
      the scorer saw: ['disliked', 'regretted', 'watching']
wrong (truth -1, score +1): Not good at all.
      the scorer saw: ['good']
wrong (truth +1, score -1): Not bad, I would watch it again.
      the scorer saw: ['bad', 'would', 'watch']
wrong (truth -1, score +2): No good scenes, no good acting.
      the scorer saw: ['good', 'scene', 'good', 'acting']
wrong (truth +1, score +0): It wasn't bad, actually quite good.
      the scorer saw: ['bad', 'actually', 'quite', 'good']
wrong (truth -1, score +2): I didn't like it, it wasn't good.
      the scorer saw: ['like', 'good']


### Step 3: answer these in your own words

**Q1. The grade.** What accuracy did the AI's pipeline score, compared with
your careful pipeline from Part 5?

The AI's pipeline scored 36.36% accuracy, while my careful pipeline scored 100%. My pipeline performed better because it preserved negation words.

**Q2. The mistakes.** Which reviews did it get wrong, and what did the
scorer actually see? Which step caused each mistake?

The AI got 7 reviews wrong. Removing stop words deleted negations like 'not' and 'no', making negative reviews appear positive. Filtering also removed contractions such as 'wasn't' and 'didn't'. Some mistakes happened because words like 'enjoyed', 'disliked', and 'regretted' were not properly lemmatized.

**Q3. The prompt.** The prompt said "remove stop words" without mentioning
negation. Rewrite it so the function you get back would score better.

Write a Python function ai_preprocess(text) using NLTK for sentiment analysis. Clean and lowercase the text while keeping apostrophes, tokenize it, lemmatize words using their parts of speech, and remove stop words except negations such as 'not', 'no', 'nor', and "n't". Return a list of tokens.

**Q4. The right tool.** If you were going to feed these reviews to a
pretrained transformer instead of a word-counting scorer, which of
the steps would you skip, and why? (Part 6.)

I would skip manual tokenization, stop-word removal, and lemmatization. A pretrained transformer has its own tokenizer and needs the original words and context to understand the meaning, especially negation.

> **The point:** the function runs and returns a tidy list of words. Only a
> test tells you that it makes **negative reviews look positive**.

---
# Going further

Optional. Nothing here is required for the lab.

| Part | Resource | Why it's worth your time |
|---|---|---|
| 1 to 5 | [NLTK documentation](https://www.nltk.org/) | The library used in Parts 2 to 5. |
| 1 to 5 | [*Natural Language Processing with Python*](https://www.nltk.org/book/) | The free NLTK book, with many more examples. |
| 1 to 5 | [spaCy](https://spacy.io/) | Another popular library, with built-in tokenization and lemmatization. |
| 6 | [Hugging Face NLP Course](https://huggingface.co/learn/nlp-course) | A free course on tokenizers, transformers and the Hub. |
| 6 | [Tokenizer summary](https://huggingface.co/docs/transformers/tokenizer_summary) | Why subword tokenizers work the way they do. |

### Optional: a ready-made model in two lines

A Hugging Face **pipeline** combines the tokenizer, a pretrained model and
the post-processing:

```python
from transformers import pipeline

classifier = pipeline("sentiment-analysis")       # downloads a model the first time
print(classifier(["Not good at all.", "I loved it and enjoyed every minute."]))
```

It needs an internet connection and downloads a few hundred MB, and this
notebook doesn't run it. It is a good one to try on **Colab**: add
`device=0` to `pipeline(...)` to use the GPU you met in Lab 9. Does it get
`Not good at all.` right, as your careful pipeline did?

---
# Before you finish

- [ ] Every self-check cell prints "correct"
- [ ] `nltk` and `transformers` installed **and** recorded in `requirements.txt`
- [ ] AI checkpoint answered in your own words
- [ ] **Restart Kernel and Run All**: the whole notebook runs with no errors
- [ ] Committed and pushed

### The habits worth keeping

1. Every cleaning rule has a cost: removing apostrophes broke `wasn't`.
2. What counts as a word is a choice: tokenization decides what later steps can see.
3. Stop-word removal can delete negation, and negation flips meaning.
4. Give the lemmatizer a part of speech, and tag the **whole** sentence first.
5. Don't trust a pipeline because it runs: build a small test and **measure** it.
6. Modern models tokenize for themselves: don't over-preprocess for them.